In [ ]:
!pip install segmentation-models

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import keras, tensorflow as tf
import keras.utils, tensorflow.keras.utils as tf_utils

# patch supaya segmentation_models tidak error
keras.utils.generic_utils = tf_utils

import segmentation_models as sm
sm.set_framework('tf.keras')
sm.framework()


# Inisialisasi dataset


In [ ]:
# ==============================================================================
# KONEKSI GOOGLE DRIVE DAN PERSIAPAN PATH
# ==============================================================================
import os # Import os module here
BASE_DATA_DIR = '/content/drive/MyDrive/Draft Skripsi/Dataset'

IMAGE_DIR = os.path.join(BASE_DATA_DIR, 'JPEGImages')
MASK_DIR = os.path.join(BASE_DATA_DIR, 'SegmentationClass')

In [ ]:
# ==============================================================================
# MEMUAT DAN MEMPROSES DATA (VERSI TENSORFLOW, DENGAN HANDLE BROWN SPOT)
# ==============================================================================

import os, cv2
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
import matplotlib.pyplot as plt

# --- KONSTANTA ---
IMG_SIZE = (384, 384)
BATCH_SIZE = 4
N_CLASSES = 5  # 0:bg, 1:leaf, 2:bacterial_blight, 3:blast, 4:brown_spot

# ==================== COLOR MAPPING (RGB setelah cvtColor) ====================
COLOR2CLASS = {
    (0, 0, 0): 0,
    (255, 0, 0): 1,   # Leaf
    (245, 128, 6): 2, # Blight
    (53, 119, 181): 3,# Blast
    (170, 170, 255): 4# Brown spot
}

IDX2NAME = {
    0: "Background",
    1: "Blast",
    2: "Bacterial Blight",
    3: "Leaf",
    4: "Brown Spot"
}

# --- SINKRONISASI FILE ---
image_files = sorted(os.listdir(IMAGE_DIR))
mask_files = sorted(os.listdir(MASK_DIR))

image_filenames_set = set(os.path.splitext(f)[0] for f in image_files)
mask_filenames_set = set(os.path.splitext(f)[0] for f in mask_files)
common_files = sorted(list(image_filenames_set.intersection(mask_filenames_set)))

image_paths, mask_paths = [], []
for fname in common_files:
    img_found, mask_found = None, None
    for ext in ['.png', '.jpg', '.jpeg']:
        if os.path.exists(os.path.join(IMAGE_DIR, fname + ext)):
            img_found = os.path.join(IMAGE_DIR, fname + ext)
        if os.path.exists(os.path.join(MASK_DIR, fname + ext)):
            mask_found = os.path.join(MASK_DIR, fname + ext)
    if img_found and mask_found:
        image_paths.append(img_found)
        mask_paths.append(mask_found)

print(f"Jumlah pasangan gambar & mask yang cocok: {len(image_paths)}")

# --- UTILITAS UNTUK CEK LABEL ---
def rgb_to_class_mask(mask_rgb, color_map):
    h, w, _ = mask_rgb.shape
    mask_class = np.zeros((h, w), dtype=np.uint8)
    for color, class_idx in color_map.items():
        mask_class[np.all(mask_rgb == color, axis=-1)] = class_idx
    return mask_class

def has_brown_spot(mask_path):
    mask_rgb = cv2.imread(mask_path)
    mask_rgb = cv2.cvtColor(mask_rgb, cv2.COLOR_BGR2RGB)
    mask_class = rgb_to_class_mask(mask_rgb, COLOR2CLASS)
    return 4 in np.unique(mask_class)

def get_dominant_class(mask_path):
    mask_rgb = cv2.imread(mask_path)
    mask_rgb = cv2.cvtColor(mask_rgb, cv2.COLOR_BGR2RGB)
    mask_class = rgb_to_class_mask(mask_rgb, COLOR2CLASS)
    vals, counts = np.unique(mask_class, return_counts=True)
    return vals[np.argmax(counts)]

# --- PISAHKAN FILE BROWN SPOT ---
brownspot_files = [(img, msk) for img, msk in zip(image_paths, mask_paths) if has_brown_spot(msk)]
print(f"Jumlah file dengan Brown Spot: {len(brownspot_files)}")

manual_test   = brownspot_files[:1]   # 1 ke Test
manual_val    = brownspot_files[1:2]  # 1 ke Val
manual_train  = brownspot_files[2:]   # sisanya ke Train

# --- DATA TANPA BROWN SPOT ---
image_paths_clean = [img for img, msk in zip(image_paths, mask_paths) if (img, msk) not in brownspot_files]
mask_paths_clean  = [msk for img, msk in zip(image_paths, mask_paths) if (img, msk) not in brownspot_files]
dominant_labels   = [get_dominant_class(m) for m in mask_paths_clean]

# --- SPLIT 80/15/5 ---
train_imgs, test_imgs, train_masks, test_masks, y_train, y_test = train_test_split(
    image_paths_clean, mask_paths_clean, dominant_labels,
    test_size=0.05, random_state=42, stratify=dominant_labels
)

train_imgs, val_imgs, train_masks, val_masks, y_train, y_val = train_test_split(
    train_imgs, train_masks, y_train,
    test_size=0.1579, random_state=42, stratify=y_train  # 15 / (100-5)
)

# --- TAMBAHKAN FILE BROWN SPOT ---
train_imgs  += [img for img, msk in manual_train]
train_masks += [msk for img, msk in manual_train]
val_imgs    += [img for img, msk in manual_val]
val_masks   += [msk for img, msk in manual_val]
test_imgs   += [img for img, msk in manual_test]
test_masks  += [msk for img, msk in manual_test]

print(f"\nTotal gambar yang digunakan: {len(image_paths)}")
print(f"Train: {len(train_imgs)}, Val: {len(val_imgs)}, Test: {len(test_imgs)}")

# ==============================================================================
#  PARSING & PIPELINE TF.DATA
# ==============================================================================

@tf.function
def parse_mask_robust(mask_rgb):
    """Konversi mask RGB → indeks kelas dengan mencari warna terdekat."""
    mask_rgb_float = tf.cast(mask_rgb, tf.float32)
    color_map = tf.constant(list(COLOR2CLASS.keys()), dtype=tf.float32)  # shape (5,3)
    mask_expanded = tf.expand_dims(mask_rgb_float, axis=2)  # (H,W,1,3)
    color_map_expanded = tf.reshape(color_map, [1,1,N_CLASSES,3])
    distances = tf.reduce_sum(tf.square(mask_expanded - color_map_expanded), axis=3)
    semantic_map = tf.argmin(distances, axis=2, output_type=tf.int32)
    return tf.cast(semantic_map, tf.uint8)

def load_and_preprocess(image_path, mask_path):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0

    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=3)
    mask = tf.image.resize(mask, IMG_SIZE, method='nearest')
    mask = parse_mask_robust(mask)
    mask = tf.one_hot(mask, depth=N_CLASSES, dtype=tf.float32)
    return image, mask

def create_dataset(images, masks, is_training=False):
    dataset = tf.data.Dataset.from_tensor_slices((images, masks))
    dataset = dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if is_training:
        dataset = dataset.shuffle(200)
    return dataset.batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

train_dataset = create_dataset(train_imgs, train_masks, is_training=True)
val_dataset   = create_dataset(val_imgs, val_masks)
test_dataset  = create_dataset(test_imgs, test_masks)

# ==============================================================================
#  CEK DISTRIBUSI LABEL
# ==============================================================================

def count_labels_tf(dataset, name):
    counts = np.zeros(N_CLASSES, dtype=np.int64)
    for _, masks in dataset:
        masks_np = tf.argmax(masks, axis=-1).numpy()  # (B,H,W)
        unique, c = np.unique(masks_np, return_counts=True)
        for u, cc in zip(unique, c):
            counts[u] += cc
    total = counts.sum()
    print(f"\nDistribusi pixel label di {name}:")
    for i, cnt in enumerate(counts):
        pct = 100.0 * cnt / total if total > 0 else 0
        print(f"  Class {i} ({IDX2NAME[i]}): {cnt} pixels ({pct:.2f}%)")

def count_files_by_dominant(imgs, masks, name):
    labels = [get_dominant_class(m) for m in masks]
    unique, counts = np.unique(labels, return_counts=True)
    print(f"\nDistribusi file dominan di {name}:")
    for u, c in zip(unique, counts):
        print(f"  Class {u} ({IDX2NAME[u]}): {c} files")

# # Hitung distribusi pixel
# count_labels_tf(train_dataset, "Train")
# count_labels_tf(val_dataset, "Validation")
# count_labels_tf(test_dataset, "Test")

# # Hitung distribusi file dominan
# count_files_by_dominant(train_imgs, train_masks, "Train")
# count_files_by_dominant(val_imgs, val_masks, "Validation")
# count_files_by_dominant(test_imgs, test_masks, "Test")


# Train

In [ ]:
import numpy as np
# Inisialisasi model UNet
model = sm.Unet('resnet34', classes=N_CLASSES, activation='softmax', encoder_weights='imagenet')

# Definisikan loss dan metrics untuk digunakan saat kompilasi dan pemuatan model
dice_loss = sm.losses.DiceLoss(class_weights=np.array([0.1, 0.2, 0.3, 0.2, 0.2]))
focal_loss = sm.losses.CategoricalFocalLoss()
total_loss = dice_loss + (1 * focal_loss)

metrics = [sm.metrics.IOUScore(threshold=0.5), sm.metrics.FScore(threshold=0.5)]

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss=total_loss, metrics=metrics)

In [ ]:
import os
import tensorflow as tf
import segmentation_models as sm

# --- METRIC IoU ---
iou = sm.metrics.IOUScore(threshold=None)

if 'model' in locals():
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=[iou]
    )

# --- SETUP FOLDER PENYIMPANAN ---
nama_model = "unet"  # ganti sesuai nama arsitektur yang dipakai
base_folder = "/content/drive/MyDrive/Draft Skripsi/Model"
folder_model_path = os.path.join(base_folder, nama_model)
os.makedirs(folder_model_path, exist_ok=True)

# --- PATH MODEL ---
EPOCHS = 200
best_model_path = os.path.join(folder_model_path, f"{nama_model}_best_val_iou_{EPOCHS}.keras")
last_model_path = os.path.join(folder_model_path, f"{nama_model}_last_{EPOCHS}.keras")

# --- CALLBACK CHECKPOINT ---
ckpt_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath=best_model_path,
    monitor='val_iou_score',
    mode='max',
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)

# --- TRAINING ---
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=[ckpt_cb]
)

# --- SIMPAN MODEL TERAKHIR ---
model.save(last_model_path)

print(f"Best model disimpan di: {best_model_path}")
print(f"Last model disimpan di: {last_model_path}")


In [ ]:
import os
import matplotlib.pyplot as plt

nama_model = "unet"
base_dir   = "/content/drive/MyDrive/Draft Skripsi/learning_curve"
save_dir   = os.path.join(base_dir, nama_model)
os.makedirs(save_dir, exist_ok=True)

# ambil riwayat training
hist = history.history
print("Keys dalam history:", hist.keys())

epochs_range = range(1, len(hist['loss'])+1)

plt.figure(figsize=(12,5))

# ---- Loss curve ----
plt.subplot(1,2,1)
plt.plot(epochs_range, hist['loss'], label="Train Loss", marker="o")
plt.plot(epochs_range, hist['val_loss'], label="Val Loss", marker="s")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Learning Curve - Loss")
plt.legend()
plt.grid(True)

# ---- IoU curve ----
# cek apakah metric bernama "iou_score" (default sm.metrics.IOUScore)
if "iou_score" in hist:
    train_iou = hist["iou_score"]
    val_iou   = hist["val_iou_score"]
elif "accuracy" in hist:  # fallback kalau pakai metric lain
    train_iou = hist["accuracy"]
    val_iou   = hist["val_accuracy"]
else:
    raise KeyError("Metric IoU tidak ditemukan di history.history")

plt.subplot(1,2,2)
plt.plot(epochs_range, train_iou, label="Train IoU", marker="o")
plt.plot(epochs_range, val_iou, label="Val IoU", marker="s")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.title("Learning Curve - IoU")
plt.legend()
plt.grid(True)

plt.tight_layout()

# === Simpan ke Google Drive ===
curve_path = os.path.join(save_dir, f"{nama_model}_learning_curve_{EPOCHS}.png")
plt.savefig(curve_path, dpi=300)
plt.show()

print(f"Learning curve disimpan ke: {curve_path}")


# Evaluate

In [ ]:
import tensorflow as tf
import numpy as np
from tqdm import tqdm

# =================== SETUP ===================
N_CLASSES = 5
MODEL_PATH = "/content/drive/MyDrive/Draft Skripsi/Model/unet/unet_best_val_iou_200.keras"

model = tf.keras.models.load_model(MODEL_PATH, compile=False)

def is_probabilities(t):
    # true jika tiap piksel menjumlah ~1 di dim kelas
    s = tf.reduce_sum(t, axis=-1)
    return bool(tf.reduce_all(tf.abs(s - 1.0) < 1e-3).numpy())

miou_metric = tf.keras.metrics.MeanIoU(num_classes=N_CLASSES)

miou_per_img, acc_per_img, losses = [], [], []

for batch in tqdm(test_dataset, desc="[Testing]"):
    x_batch, y_batch = batch                      # y_batch bisa (B,H,W) atau (B,H,W,C)
    preds = model(x_batch, training=False)        # (B,H,W,C)

    # Deteksi apakah preds sudah probabilitas
    preds_are_prob = is_probabilities(preds)
    from_logits    = not preds_are_prob

    # Deteksi bentuk label: one-hot vs indeks
    labels_are_onehot = (y_batch.shape.ndims == 4 and y_batch.shape[-1] == N_CLASSES)

    # ----- Siapkan loss function yang sesuai -----
    if labels_are_onehot:
        loss_fn = tf.keras.losses.CategoricalCrossentropy(from_logits=from_logits)
        batch_loss = loss_fn(y_batch, preds)
        y_true_idx = tf.argmax(y_batch, axis=-1)          # (B,H,W)
    else:
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=from_logits)
        batch_loss = loss_fn(y_batch, preds)
        y_true_idx = y_batch                               # (B,H,W)

    losses.append(float(batch_loss.numpy()))

    # Prediksi indeks kelas
    y_pred_idx = tf.argmax(preds, axis=-1)                 # (B,H,W)

    # ----- Loop per image -----
    B = tf.shape(y_pred_idx)[0]
    for i in range(int(B.numpy())):
        yt = y_true_idx[i]
        yp = y_pred_idx[i]

        # Accuracy per image
        acc = tf.reduce_mean(tf.cast(tf.equal(yt, yp), tf.float32)).numpy()
        acc_per_img.append(float(acc))

        # mIoU per image
        miou_metric.reset_state()
        miou_metric.update_state(yt, yp)   # butuh label indeks
        miou_per_img.append(float(miou_metric.result().numpy()))

# =================== HASIL AKHIR ===================
mean_loss = float(np.mean(losses)) if losses else 0.0
mean_miou = float(np.mean(miou_per_img)) if miou_per_img else 0.0
mean_acc  = float(np.mean(acc_per_img))  if acc_per_img  else 0.0

print("\n== Evaluation Results (Per-Image Averaged) ==")
print(f"Loss (avg batch): {mean_loss:.4f}")
print(f"Mean mIoU (per-image): {mean_miou:.4f}")
print(f"Mean Accuracy (per-image): {mean_acc:.4f}")


# Analisis Keparahan di Data Test

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# ================== SETUP FOLDER OUTPUT ==================
nama_model = "unet"   # ganti sesuai model

# ✅ ubah EPOCHS jadi string
hasil_dir = os.path.join(
    "/content/drive/MyDrive/Draft Skripsi/hasil_analisis",
    nama_model,
    f"epoch_{EPOCHS}"   # format string, aman
)
os.makedirs(hasil_dir, exist_ok=True)

# ================== COLOR MAPPING (RGB) ==================
COLOR2CLASS = {
    (0, 0, 0): 0,
    (255, 0, 0): 1,   # Leaf
    (245, 128, 6): 2, # Bacterial Blight
    (53, 119, 181): 3,# Blast
    (170, 170, 255): 4# Brown Spot
}

IDX2NAME = {
    0: "Background",
    1: "Leaf",
    2: "Bacterial Blight",
    3: "Blast",
    4: "Brown Spot"
}

CLASS_COLORS = {v: k for k, v in COLOR2CLASS.items()}

def decode_segmap(mask, n_classes=len(CLASS_COLORS), class_colors=CLASS_COLORS):
    """Konversi mask indeks ke RGB"""
    h, w = mask.shape
    output = np.zeros((h, w, 3), dtype=np.uint8)
    for i in range(n_classes):
        color = class_colors.get(i, (0, 0, 0))
        output[mask == i] = color
    return output

# ================== SEVERITY (IRRI SES) ==================
def severity_from_pred(pred_idx, gt_idx=None,
                       idx_leaf=3, idx_blight=2, idx_blast=1, idx_brown=4,
                       use_gt_leaf=True):
    pixel_blight = int(np.sum(pred_idx == idx_blight))
    pixel_blast  = int(np.sum(pred_idx == idx_blast))
    pixel_brown  = int(np.sum(pred_idx == idx_brown))
    total_penyakit = pixel_blight + pixel_blast + pixel_brown

    if use_gt_leaf and gt_idx is not None:
        pixel_leaf = int(np.sum(gt_idx == idx_leaf))
    else:
        pixel_leaf = int(np.sum(pred_idx == idx_leaf))

    total_daun = pixel_leaf + total_penyakit
    persen = 0.0 if total_daun <= 0 else 100.0 * total_penyakit / total_daun

    if persen < 1:
        tingkat = "Gejala Awal (SES 1–3)"
    elif persen < 10:
        tingkat = "Ringan (SES 4–5)"
    elif persen < 25:
        tingkat = "Sedang (SES 6)"
    elif persen < 50:
        tingkat = "Parah (SES 7)"
    else:
        tingkat = "Sangat Parah (SES 8–9)"

    penyakit = []
    if pixel_blight > 0: penyakit.append("Bacterial Blight")
    if pixel_blast  > 0: penyakit.append("Blast")
    if pixel_brown  > 0: penyakit.append("Brown Spot")
    if not penyakit:
        tingkat = "Sehat (SES 0)"
        penyakit_str = "Tidak ada penyakit terdeteksi"
    else:
        penyakit_str = ", ".join(penyakit)

    return persen, tingkat, penyakit_str

# ================== EVALUASI MODEL ==================
all_ious = {i: [] for i in range(len(IDX2NAME))}
correct_pixels, total_pixels = 0, 0

# buka file log tunggal di sub-folder model
log_path = os.path.join(hasil_dir, "hasil_evaluasi.txt")
f_log = open(log_path, "w", encoding="utf-8")

sample_id = 0
for images_b, masks_b in test_dataset:
    logits_b = model.predict(images_b, verbose=0)
    pred_idx_b = np.argmax(logits_b, axis=-1)
    true_idx_b = np.argmax(masks_b.numpy(), axis=-1)

    correct_pixels += np.sum(pred_idx_b == true_idx_b)
    total_pixels   += np.prod(true_idx_b.shape)

    # IoU tiap kelas
    for cls in range(len(IDX2NAME)):
        inter = np.logical_and(pred_idx_b == cls, true_idx_b == cls).sum()
        union = np.logical_or(pred_idx_b == cls, true_idx_b == cls).sum()
        if union > 0:
            all_ious[cls].append(inter / union)

    # simpan analisis tiap sample
    B = images_b.shape[0]
    for k in range(B):
        sample_id += 1
        img    = (images_b[k].numpy() * 255).astype(np.uint8)
        gt_idx = true_idx_b[k]
        pr_idx = pred_idx_b[k]

        persen_k, tingkat_k, penyakit_k = severity_from_pred(pr_idx, gt_idx, use_gt_leaf=True)

        # tulis ke file log
        f_log.write(f"\n=== Sampel {sample_id} ===\n")
        f_log.write(f"Penyakit Terdeteksi      : {penyakit_k}\n")
        f_log.write(f"Persentase Area Penyakit : {persen_k:.2f}%\n")
        f_log.write(f"Tingkat Keparahan        : {tingkat_k}\n")
        f_log.write(f"Unique GT mask values : {np.unique(gt_idx)}\n")
        f_log.write(f"Unique Pred mask values: {np.unique(pr_idx)}\n")

        # simpan gambar tiap sample
        plt.figure(figsize=(14,4))
        plt.subplot(1,3,1); plt.title(f"Image [{sample_id}]")
        plt.imshow(img); plt.axis("off")
        plt.subplot(1,3,2); plt.title("Ground Truth")
        plt.imshow(decode_segmap(gt_idx)); plt.axis("off")
        plt.subplot(1,3,3); plt.title("Prediction")
        plt.imshow(decode_segmap(pr_idx)); plt.axis("off")

        save_path = os.path.join(hasil_dir, f"visualisasi_sample_{sample_id}.png")
        plt.savefig(save_path, dpi=150)
        plt.close()

# ================== HASIL GLOBAL ==================
accuracy = correct_pixels / total_pixels
miou = np.nanmean([np.mean(v) if len(v)>0 else np.nan for v in all_ious.values()])

f_log.write("\n\n== Evaluation Results (Global) ==\n")
f_log.write(f"Accuracy : {accuracy:.4f}\n")
f_log.write(f"mIoU     : {miou:.4f}\n")

f_log.write("IoU per class:\n")
for i, cname in IDX2NAME.items():
    if len(all_ious[i]) > 0:
        val = float(np.mean(all_ious[i]))
    else:
        val = 0.0
    f_log.write(f"  {cname:>16}: {val:.4f}\n")

f_log.close()

print(f"\n✅ Semua hasil disimpan di folder: {hasil_dir}")
print(f"- Ringkasan + analisis tiap data: {log_path}")
print(f"- Visualisasi per sample: {hasil_dir}/visualisasi_sample_*.png")


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tqdm import tqdm

# ================== SETUP FOLDER OUTPUT ==================
nama_model = "unet"   # ganti sesuai model
EPOCHS = 200          # ganti sesuai epoch kamu

hasil_dir = os.path.join(
    "/content/drive/MyDrive/Draft Skripsi/hasil_analisisGT",
    nama_model,
    f"epoch_{EPOCHS}"
)
os.makedirs(hasil_dir, exist_ok=True)

# ================== COLOR MAPPING (RGB) ==================
COLOR2CLASS = {
    (0, 0, 0): 0,           # Background
    (255, 0, 0): 1,         # Leaf
    (245, 128, 6): 2,       # Bacterial Blight
    (53, 119, 181): 3,      # Blast
    (170, 170, 255): 4      # Brown Spot
}
IDX2NAME = {
    0: "Background",
    1: "Leaf",
    2: "Bacterial Blight",
    3: "Blast",
    4: "Brown Spot"
}
CLASS_COLORS = {v: k for k, v in COLOR2CLASS.items()}

def decode_segmap(mask, n_classes=len(CLASS_COLORS), class_colors=CLASS_COLORS):
    """Konversi mask indeks ke RGB"""
    h, w = mask.shape
    out = np.zeros((h, w, 3), dtype=np.uint8)
    for i in range(n_classes):
        color = class_colors.get(i, (0, 0, 0))
        out[mask == i] = color
    return out

# ================== FUNGSI KEPARAHAN ==================
def severity_from_gt(gt_idx,
                     idx_leaf=1, idx_blight=2, idx_blast=3, idx_brown=4):
    """Hitung keparahan penyakit berdasarkan Ground Truth"""
    pixel_leaf   = np.sum(gt_idx == idx_leaf)
    pixel_blight = np.sum(gt_idx == idx_blight)
    pixel_blast  = np.sum(gt_idx == idx_blast)
    pixel_brown  = np.sum(gt_idx == idx_brown)
    total_penyakit = pixel_blight + pixel_blast + pixel_brown
    total_daun = pixel_leaf + total_penyakit

    persen = 0.0 if total_daun == 0 else 100.0 * total_penyakit / total_daun

    # Klasifikasi IRRI SES
    if persen < 1:
        tingkat = "Gejala Awal (SES 1–3)"
    elif persen < 10:
        tingkat = "Ringan (SES 4–5)"
    elif persen < 25:
        tingkat = "Sedang (SES 6)"
    elif persen < 50:
        tingkat = "Parah (SES 7)"
    else:
        tingkat = "Sangat Parah (SES 8–9)"

    penyakit = []
    if pixel_blight > 0: penyakit.append("Bacterial Blight")
    if pixel_blast  > 0: penyakit.append("Blast")
    if pixel_brown  > 0: penyakit.append("Brown Spot")
    penyakit_str = ", ".join(penyakit) if penyakit else "Tidak ada penyakit terdeteksi"

    return persen, tingkat, penyakit_str, pixel_leaf, total_penyakit


# ================== ANALISIS GROUND TRUTH ==================
log_path = os.path.join(hasil_dir, "hasil_analisis_groundtruth.txt")
f_log = open(log_path, "w", encoding="utf-8")

sample_id = 0
total_persen = []

for images_b, masks_b in tqdm(test_dataset, desc="Analisis Ground Truth"):
    imgs = images_b.numpy()
    gts  = np.argmax(masks_b.numpy(), axis=-1)

    for k in range(imgs.shape[0]):
        sample_id += 1
        img = (imgs[k] * 255).astype(np.uint8)
        gt_idx = gts[k]

        persen, tingkat, penyakit, daun_px, sakit_px = severity_from_gt(gt_idx)
        total_persen.append(persen)

        # tulis hasil ke file
        f_log.write(f"\n=== Ground Truth Sampel {sample_id} ===\n")
        f_log.write(f"Penyakit terdeteksi     : {penyakit}\n")
        f_log.write(f"Luas daun               : {daun_px} piksel\n")
        f_log.write(f"Luas area penyakit      : {sakit_px} piksel\n")
        f_log.write(f"Persentase keparahan    : {persen:.2f}%\n")
        f_log.write(f"Tingkat keparahan (SES) : {tingkat}\n")

        # deskripsi otomatis
        if persen >= 50:
            f_log.write("  → Kondisi sangat parah, sebagian besar daun terinfeksi.\n")
        elif persen >= 25:
            f_log.write("  → Infeksi cukup luas, gejala terlihat jelas.\n")
        elif persen >= 10:
            f_log.write("  → Gejala sedang, mulai menyebar pada permukaan daun.\n")
        elif persen > 0:
            f_log.write("  → Gejala ringan, hanya sebagian kecil area daun terinfeksi.\n")
        else:
            f_log.write("  → Tidak ditemukan gejala penyakit (daun sehat).\n")

        f_log.write("-"*60 + "\n")

        # Simpan visualisasi GT
        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1); plt.title("Citra Asli")
        plt.imshow(img.astype(np.uint8)); plt.axis("off")
        plt.subplot(1, 2, 2); plt.title("Ground Truth Mask")
        plt.imshow(decode_segmap(gt_idx)); plt.axis("off")

        save_path = os.path.join(hasil_dir, f"gt_visual_{sample_id}.png")
        plt.savefig(save_path, dpi=150)
        plt.close()

# ================== RINGKASAN GLOBAL ==================
rata2 = np.mean(total_persen)
min_persen = np.min(total_persen)
max_persen = np.max(total_persen)

f_log.write("\n\n== RINGKASAN GLOBAL (GROUND TRUTH) ==\n")
f_log.write(f"Rata-rata keparahan : {rata2:.2f}%\n")
f_log.write(f"Minimum keparahan   : {min_persen:.2f}%\n")
f_log.write(f"Maksimum keparahan  : {max_persen:.2f}%\n")
f_log.close()

print(f"\n✅ Semua hasil ANALISIS GROUND TRUTH disimpan di:")
print(f"{hasil_dir}")
print(f"- File teks: {log_path}")
print(f"- Gambar visualisasi: gt_visual_*.png")
